IMPORTING LIBRARIES

In [16]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Bidirectional

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [17]:
df = pd.read_csv("/content/Emotion_Cassify_Data.csv")

In [18]:
df.head()

,Comment,Emotion
0,i seriously hate one subject to death but now ...,fear
1,im so full of life i feel appalled,anger
2,i sit here to write i start to dig out my feel...,fear
3,ive been really angry with r and i feel like a...,joy
4,i feel suspicious if there is no one outside l...,fear


In [19]:
X = df['Comment']
y = df['Emotion']

In [20]:
#LABEL ENCODING
encoder = LabelEncoder()

y = encoder.fit_transform(y)

In [21]:
#TOKENIZATION
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)

In [22]:
#PADDING
max_length = 100

X_pad = pad_sequences(
    X_seq,
    maxlen=max_length,
    padding='post'
)

In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad,
    y,
    test_size=0.2,
    random_state=42)

In [24]:
#LSTM MODEL
model = Sequential()

model.add( Embedding(
        input_dim=5000,
        output_dim=128,
        input_length=max_length))

model.add(LSTM(64))

model.add(Dense(
        len(np.unique(y)),
        activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
#BUILDING MODEL
model.build(input_shape=(None, max_length))

In [26]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 100, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 689,603 (2.63 MB)

 Trainable params: 689,603 (2.63 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

In [28]:
#TRAINING MODEL
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test))

Epoch 1/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 9s 52ms/step - accuracy: 0.3169 - loss: 1.1000 - val_accuracy: 0.3502 - val_loss: 1.0984
Epoch 2/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 8s 53ms/step - accuracy: 0.3279 - loss: 1.0987 - val_accuracy: 0.3199 - val_loss: 1.1012
Epoch 3/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 7s 49ms/step - accuracy: 0.3455 - loss: 1.0985 - val_accuracy: 0.3300 - val_loss: 1.1019
Epoch 4/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 8s 52ms/step - accuracy: 0.3386 - loss: 1.0989 - val_accuracy: 0.3199 - val_loss: 1.0994
Epoch 5/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 7s 47ms/step - accuracy: 0.3371 - loss: 1.0986 - val_accuracy: 0.3300 - val_loss: 1.0999


In [29]:
loss, accuracy = model.evaluate(X_test, y_test)

print("Loss:", loss)
print("Accuracy:", accuracy)

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.3300 - loss: 1.0999
Loss: 1.0998563766479492
Accuracy: 0.32996633648872375


In [30]:
#BILSTM MODEL
bilstm_model = Sequential()

bilstm_model.add(
    Embedding(
        input_dim=5000,
        output_dim=128,
        input_length=max_length))

bilstm_model.add(
    Bidirectional(
        LSTM(64)))

bilstm_model.add(
    Dense(len(np.unique(y)),
        activation='softmax'))

In [31]:
bilstm_model.build(input_shape=(None, max_length))

In [32]:
bilstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 100, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 128)            │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 739,203 (2.82 MB)

 Trainable params: 739,203 (2.82 MB)

 Non-trainable params: 0 (0.00 B)

In [33]:
bilstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

In [34]:
history_bilstm = bilstm_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test))

Epoch 1/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 16s 91ms/step - accuracy: 0.5534 - loss: 0.9076 - val_accuracy: 0.8721 - val_loss: 0.3963
Epoch 2/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 14s 92ms/step - accuracy: 0.9360 - loss: 0.1891 - val_accuracy: 0.9268 - val_loss: 0.2057
Epoch 3/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 20s 89ms/step - accuracy: 0.9779 - loss: 0.0713 - val_accuracy: 0.9436 - val_loss: 0.1829
Epoch 4/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 22s 98ms/step - accuracy: 0.9867 - loss: 0.0387 - val_accuracy: 0.9487 - val_loss: 0.1715
Epoch 5/5
149/149 ━━━━━━━━━━━━━━━━━━━━ 20s 92ms/step - accuracy: 0.9895 - loss: 0.0340 - val_accuracy: 0.9200 - val_loss: 0.2906


In [35]:
loss, accuracy = bilstm_model.evaluate(X_test, y_test)

print("BiLSTM Loss:", loss)
print("BiLSTM Accuracy:", accuracy)

38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9200 - loss: 0.2906
BiLSTM Loss: 0.2906484305858612
BiLSTM Accuracy: 0.9200336933135986


In [36]:
sample_texts = ["I am feeling very happy today",
                "I am going to picnic today",
                "I am scared about tomorrow exam",
                "I am feeling very angry today",
                "i sit here to write i start to dig out my feelings and i think that i am afraid to accept the possibility that he might not make it"]

In [37]:
for text in sample_texts:

    # Convert text to sequence
    seq = tokenizer.texts_to_sequences([text])

    # Padding
    padded = pad_sequences(
        seq,
        maxlen=max_length,
        padding='post')

    # Prediction
    prediction = bilstm_model.predict(padded)

    # Highest probability class
    predicted_class = np.argmax(prediction)

    # Convert number to emotion name
    predicted_emotion = encoder.inverse_transform(
        [predicted_class])

    print("Text :", text)

    print("Predicted Emotion :", predicted_emotion[0])

    print("-----------------------------------")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 260ms/step
Text : I am feeling very happy today
Predicted Emotion : joy
-----------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Text : I am going to picnic today
Predicted Emotion : joy
-----------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
Text : I am scared about tomorrow exam
Predicted Emotion : fear
-----------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
Text : I am feeling very angry today
Predicted Emotion : anger
-----------------------------------
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step
Text : i sit here to write i start to dig out my feelings and i think that i am afraid to accept the possibility that he might not make it
Predicted Emotion : fear
-----------------------------------
